# MOSS-Image on a free Colab T4
Text-to-image with [MOSS-Image-SFT](https://huggingface.co/OpenMOSS-Team/MOSS-Image-SFT) (5B parameters, Apache-2.0, strong at text inside images). Its weights are 27 GB, so this notebook stores the image model as **int8** and the text encoder as **4-bit** to fit a free T4 (16 GB).

**Before you start:** Runtime → Change runtime type → **T4 GPU**. The first run downloads about 25 GB (around 15 minutes), then loading takes about 2.5 minutes and each 768×768 picture about 2 minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Install
`optimum-quanto` stores the image model in int8 and `bitsandbytes` stores the text encoder in 4-bit. Colab's own `torchao` is older than the current `diffusers` expects and breaks `import diffusers`, so it is removed (nothing here uses it).

In [ ]:
!pip -q install optimum-quanto bitsandbytes
!pip -q uninstall -y torchao

## 2. Download the model (about 25 GB, once per session)

In [ ]:
from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download("OpenMOSS-Team/MOSS-Image-SFT", local_dir="/content/moss-sft")
print("model files in", MODEL_DIR)

## 3. Load it in low-memory form
The T4 has no native bfloat16, so everything runs in float16. Loading takes about 2.5 minutes and leaves about 11.5 GB on the GPU.

In [ ]:
import time, torch
from diffusers import DiffusionPipeline, QuantoConfig
from diffusers.quantizers import PipelineQuantizationConfig
from transformers import BitsAndBytesConfig

DT = torch.float16
quant = PipelineQuantizationConfig(quant_mapping={
    "transformer": QuantoConfig(weights_dtype="int8"),
    "text_encoder": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DT),
})
t = time.time()
pipe = DiffusionPipeline.from_pretrained(
    MODEL_DIR, custom_pipeline="pipeline_moss_image", trust_remote_code=True,
    torch_dtype={"default": DT, "vae": torch.float32}, quantization_config=quant)
pipe.scheduler = type(pipe.scheduler).from_config(pipe.scheduler.config)
pipe.to("cuda")
print(f"loaded in {time.time() - t:.0f}s, {torch.cuda.memory_allocated() / 1e9:.1f} GB on the GPU")

## 4. Make a picture
`size` must be a multiple of 16. **768** is the safe choice on a free T4 (about 2 minutes, 12.9 GB peak). **1024** works but needs 14 GB at peak and takes about 4 minutes. `steps=30` is what was tested here, on flat mascot art; the model card's default is 50, which I did not try, so raise it if a picture looks rough.

In [ ]:
from IPython.display import display

NEGATIVE = "realistic, photo, 3d, shading, gradient, watermark, extra limbs"

def draw(prompt, size=768, steps=30, seed=42, negative=NEGATIVE):
    t = time.time(); torch.cuda.reset_peak_memory_stats()
    img = pipe(prompt, negative_prompt=negative, height=size, width=size, num_inference_steps=steps,
               guidance_scale=3.0, generator=torch.Generator("cpu").manual_seed(seed)).images[0]
    print(f"{time.time() - t:.0f}s, peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    return img

MARKO = ("a baby markhor mountain goat mascot standing upright on two legs, full body, front view, round fluffy turquoise-teal fur, "
         "flat cream face mask with a tiny nose and a small cat-like smile, big round dark navy eyes with white highlights, cream inner ears "
         "pointing sideways, two tall twisted corkscrew spiral saffron-orange horns, small beard tuft, cream round belly, little saffron hooves, "
         "thick dark navy outline, solid flat colours, plain white background")

img = draw("Marko, waving one hand. A " + MARKO)
display(img)
img.save("marko_wave.png")

## 5. Text inside the picture
This is what MOSS-Image is built for. English text was spelled correctly in both of my tests. Urdu script looked convincing but only the first word of my test sign was right, so check Urdu spelling yourself.

In [ ]:
img = draw('A baby markhor mascot named Marko holding up a cream sign with large bold English text written on it: "READ URDU WITH MARKO". '
           'full body, front view, round fluffy turquoise-teal fur, cream face, big navy eyes, two tall saffron corkscrew horns, '
           'thick dark navy outline, solid flat colours, plain white background')
display(img)
img.save("marko_sign_english.png")

## 6. Your own prompt
Change the text and run the cell. Prompts can be plain sentences, tag lists, or a JSON layout that places text and objects (see the [MOSS-Image user guide](https://github.com/OpenMOSS/MOSS-Image/blob/main/USERGUIDE.md)).

In [ ]:
PROMPT = "A cozy village school classroom with a chalkboard that says \"WELCOME\", flat vector illustration, warm colours"
img = draw(PROMPT, size=768, steps=30, seed=7)
display(img)
img.save("mine.png")